# genesis-fin fin-v1 on Colab: baseline → QLoRA → compare

Runs on one Colab GPU. **Runtime → Change runtime type → L4 GPU** (24 GB) is the target; A100 is faster. A free T4 (16 GB) also works for the 4B model: precision switches to fp16 automatically (T4 has no native bf16); training takes roughly 2–3× longer than on an L4.

**No Google Drive.** Data and results live on private Hugging Face repos, so a restarted session needs no permission popup:

- Colab **Secrets** must hold `HF_TOKEN` (a token with write access), with *Notebook access* switched on for this notebook.
- A private dataset repo `<you>/genesis-fin-v1-data` holds `fin_v1.zip` (made on your PC, see `docs/10_fin_ko.md`).
- Checkpoints and reports go to the private model repo `<you>/genesis-fin-v1-runs` (created for you if missing). A background thread pushes the run folder every 10 minutes.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code, packages and data

In [ ]:
%cd /content
!rm -rf /content/GENESIS-10T && git clone -q https://github.com/Joontark-kor/GENESIS-10T.git /content/GENESIS-10T
%cd /content/GENESIS-10T
!pip install -q -e ".[fin,qlora]"
# Colab ships an old torchao that makes peft fail when loading an adapter (needs >= 0.16): remove it
!pip uninstall -y -q torchao
from google.colab import userdata
from genesis import hubsync
TOKEN = userdata.get("HF_TOKEN")   # Colab Secrets; the token is never printed or saved to a file
USER = hubsync.username(TOKEN)
DATA_REPO, RUN_REPO = f"{USER}/genesis-fin-v1-data", f"{USER}/genesis-fin-v1-runs"
hubsync.fetch_file(DATA_REPO, "fin_v1.zip", "/content", TOKEN)
!mkdir -p data/applied && unzip -o -q /content/fin_v1.zip -d data/applied
!wc -l data/applied/fin_v1/sft_train.jsonl data/applied/fin_v1/sft_val.jsonl data/applied/fin_v1/eval_test.jsonl


## 2. Settings

- `MODEL`: start with a 4B model; switch to an 8B model once everything works. Check the license on the model card.
- `GPU_HOUR_PRICE`: your hourly GPU price, so reports show cost per 1,000 questions (Colab compute units can be converted to an hourly figure).


In [ ]:
MODEL = "Qwen/Qwen3-4B"
TEMPLATE = '{"enable_thinking": false}'   # Qwen3: answer directly, no thinking block
GPU_HOUR_PRICE = 1.0                     # your price per GPU hour (any currency)
RUN = "fin-v1-qwen3-4b"
import os; os.environ.update(MODEL=MODEL, TK=TEMPLATE, PRICE=str(GPU_HOUR_PRICE), RUN=RUN)

In [ ]:
# Restore a previous run folder (checkpoints) from the private repo, then keep pushing it every 10 minutes
from pathlib import Path
run_dir = Path(f"runs/applied/{RUN}")
restored = hubsync.sync_down(RUN_REPO, run_dir, TOKEN)
os.environ["RESUME"] = "--resume" if (restored and any(run_dir.glob("checkpoint-*"))) else ""
uploader = hubsync.Uploader(RUN_REPO, run_dir, TOKEN, every=600)
uploader.start()
print("restored:", restored, "| resume flag:", os.environ["RESUME"] or "(fresh run)")


## 3. Baseline: the base model before training

In [ ]:
!python -m genesis.harness.evaluate run --model "$MODEL" --template-kwargs "$TK" \
    --data data/applied/fin_v1/eval_test.jsonl --train-data data/applied/fin_v1/sft_train.jsonl --name base --gpu-hour-price "$PRICE" \
    --batch-size 16 --out reports/applied/base.json

## 4. QLoRA fine-tuning (about 1–2 hours on an L4 for the 4B model; the log shows progress)

In [ ]:
!python -m genesis.adapt.finetune --base-model "$MODEL" --template-kwargs "$TK" --qlora $RESUME \
    --gradient-checkpointing --train data/applied/fin_v1/sft_train.jsonl \
    --val data/applied/fin_v1/sft_val.jsonl --output runs/applied/$RUN \
    --epochs 2 --max-length 1024 --batch-size 4 --grad-accum 4 --eval-steps 100 --save-steps 100


If the session disconnects during training: run sections 1–2 again. Section 2 restores the checkpoints from the private repo and sets `--resume` by itself; then run the training cell.

## 5. Evaluate the tuned model and compare

In [ ]:
!python -m genesis.harness.evaluate run --model "$MODEL" --adapter runs/applied/$RUN/adapter \
    --template-kwargs "$TK" --data data/applied/fin_v1/eval_test.jsonl --train-data data/applied/fin_v1/sft_train.jsonl --name $RUN \
    --gpu-hour-price "$PRICE" --batch-size 16 --out reports/applied/$RUN.json
!python -m genesis.harness.evaluate compare reports/applied/base.json reports/applied/$RUN.json

## 6. Save results to the private Hugging Face repo

In [ ]:
uploader.stop()   # final push of the run folder
hubsync.sync_up(RUN_REPO, "reports", TOKEN, path_in_repo="reports")
print("saved to", RUN_REPO, "| background pushes:", uploader.pushes, "failures:", uploader.failures, uploader.last_error)


## 7. A few answers to read yourself

Accuracy alone can hide formatting problems. Read some wrong answers before trusting the number.

In [ ]:
import json
rep = json.load(open(f"reports/applied/{RUN}.json"))
wrong = [r for r in rep["rows"] if not r["correct"]][:5]
for r in wrong:
    print(r["kind"], "gold:", r["gold"], "pred:", r["prediction"]); print(r["output"][:300]); print("-" * 60)